In [ ]:
import pandas as pd
import os

In [ ]:
# List of years to process
years = [
    "02-03", "03-04", "04-05", "05-06", "06-07", "07-08", "08-09", "09-10"
]

for year in years:
    filename = f"{year}_standings.csv"
    
    # Load CSV with multi-level headers
    df_raw = pd.read_csv(filename, header=[0, 1])
    
    # Fix the first three columns
    cols = list(df_raw.columns)
    cols[0] = ('', 'Rank')
    cols[1] = ('', 'School')
    cols[2] = ('', 'Total')
    df_raw.columns = pd.MultiIndex.from_tuples(cols)
    
    # Fix Unnamed columns by carrying forward sport name
    cleaned_columns = []
    prev_sport = ''
    for sport, metric in df_raw.columns:
        sport = sport.strip() if isinstance(sport, str) else ''
        metric = metric.strip() if isinstance(metric, str) else ''
        
        if not sport or sport.startswith('Unnamed'):
            sport = prev_sport
        else:
            prev_sport = sport
            
        cleaned_columns.append((sport, metric))
    
    # Reassign cleaned columns
    df_raw.columns = pd.MultiIndex.from_tuples(cleaned_columns)
    
    # Save back to the same file (overwrite)
    df_raw.to_csv(filename, index=False)
    
    print(f"Processed {filename}")

In [ ]:
# List of updated year files
years = ["10-11", "11-12"]

for year in years:
    filename = f"{year}_standings.csv"
    
    # Load with two header rows
    df_raw = pd.read_csv(filename, header=[0, 1])
    
    # Fix the first five "prefix" columns
    cols = list(df_raw.columns)
    cols[0] = ('', 'Rank')
    cols[1] = ('', 'Conference')
    cols[2] = ('', 'Division')
    cols[3] = ('', 'School')
    cols[4] = ('', 'Total')
    
    # Reassign MultiIndex columns
    df_raw.columns = pd.MultiIndex.from_tuples(cols)
    
    # Fix Unnamed columns by propagating sport names
    cleaned_columns = []
    prev_sport = ''
    
    for sport, metric in df_raw.columns:
        sport = sport.strip() if isinstance(sport, str) else ''
        metric = metric.strip() if isinstance(metric, str) else ''
        
        if not sport or sport.startswith('Unnamed'):
            sport = prev_sport
        else:
            prev_sport = sport
        
        cleaned_columns.append((sport, metric))
    
    # Reassign cleaned columns
    df_raw.columns = pd.MultiIndex.from_tuples(cleaned_columns)
    
    # Save back to file (or change filename to avoid overwrite)
    df_raw.to_csv(filename, index=False)
    
    print(f"Processed {filename}")

In [ ]:
# List of year strings for 2012–13 through 2023–24
years = [
    "12-13", "13-14", "14-15", "15-16", "16-17", "17-18",
    "18-19", "20-21", "21-22", "22-23", "23-24"
]

for year in years:
    filename = f"{year}_standings.csv"
    
    # Load CSV with two header rows
    df_raw = pd.read_csv(filename, header=[0, 1])
    
    # Rename the first five columns
    cols = list(df_raw.columns)
    cols[0] = ('', 'Rank')
    cols[1] = ('', 'School')
    cols[2] = ('', 'Conference')
    cols[3] = ('', 'Division')
    cols[4] = ('', 'Total')
    
    df_raw.columns = pd.MultiIndex.from_tuples(cols)
    
    # Fix sport headers by carrying forward the last known sport
    cleaned_columns = []
    prev_sport = ''
    
    for sport, metric in df_raw.columns:
        sport = sport.strip() if isinstance(sport, str) else ''
        metric = metric.strip() if isinstance(metric, str) else ''
        
        if not sport or sport.startswith('Unnamed'):
            sport = prev_sport
        else:
            prev_sport = sport
        
        cleaned_columns.append((sport, metric))
    
    df_raw.columns = pd.MultiIndex.from_tuples(cleaned_columns)
    
    # Save the cleaned file
    df_raw.to_csv(filename, index=False)
    
    print(f"Processed {filename}")

In [ ]:
# Years to process
years = [
    "12-13", "13-14", "14-15", "15-16", "16-17", "17-18",
    "18-19", "20-21", "21-22", "22-23", "23-24"
]

# Known conferences
conferences = [
    "ACC", "Big East", "Big South", "Big Sky", "Big Ten", "Big 12", "CAA", "Conference USA", 
    "Horizon League", "Ivy League", "MAAC", "MAC", "MEAC", "Missouri Valley", "Mountain West", 
    "Northeast", "Ohio Valley", "OVC", "Pac-12", "Patriot", "SEC", "Southern", "Southland", 
    "Summit", "Sun Belt", "SWAC", "WCC", "America East", "Atlantic 10", "Atlantic Sun", 
    "WAC", "SoCon"
]
conf_set = set(c.lower() for c in conferences)

# Column identifiers
prefix_names = ["Rank", "School", "Conference", "Division", "Total"]

for year in years:
    filename = f"{year}_standings.csv"
    
    # Step 1: Load with two headers
    df_raw = pd.read_csv(filename, header=[0, 1])
    
    # Step 2: Rename first five columns
    cols = list(df_raw.columns)
    for i in range(5):
        cols[i] = ('', prefix_names[i])
    df_raw.columns = pd.MultiIndex.from_tuples(cols)
    
    # Step 3: Clean sport headers (fill forward sport names)
    cleaned_columns = []
    prev_sport = ''
    for sport, metric in df_raw.columns:
        sport = sport.strip() if isinstance(sport, str) else ''
        metric = metric.strip() if isinstance(metric, str) else ''
        if not sport or sport.startswith("Unnamed"):
            sport = prev_sport
        else:
            prev_sport = sport
        cleaned_columns.append((sport, metric))
    df_raw.columns = pd.MultiIndex.from_tuples(cleaned_columns)

    # Step 4: Fix Conference if missing and stuck on School
    school_col = ('', 'School')
    conf_col = ('', 'Conference')

    def fix_school_conf(row):
        school_val = row[school_col]
        conf_val = row[conf_col]
        
        if pd.isna(conf_val) and isinstance(school_val, str):
            words = school_val.strip().split()
            for i in range(1, min(4, len(words))):  # Try 1- to 3-word suffixes
                candidate = ' '.join(words[-i:]).lower()
                if candidate in conf_set:
                    return pd.Series({
                        school_col: ' '.join(words[:-i]),
                        conf_col: ' '.join(words[-i:])
                    })
        return pd.Series({school_col: school_val, conf_col: conf_val})

    if school_col in df_raw.columns and conf_col in df_raw.columns:
        fixed = df_raw[[school_col, conf_col]].apply(fix_school_conf, axis=1)
        df_raw[school_col] = fixed[school_col]
        df_raw[conf_col] = fixed[conf_col]

    # Step 5: Save the corrected file
    df_raw.to_csv(filename, index=False)
    print(f"Fixed and saved: {filename}")